# VoxMeet Community-1 worker — Kaggle
Keep this notebook PRIVATE: the connection cell contains a temporary access key.
Enable GPU, Internet, and the HF_TOKEN secret. Accept Community-1 access on Hugging Face.
Run setup once. If packages changed, restart the session and run ONLY the worker cell.
While the worker runs, submit recordings from the demo; no per-recording notebook transfer is needed.
Audio travels through Cloudflare to your Kaggle session. Connection expires after 8 hours; Kaggle can end sessions earlier.
This runs inference with the original Community-1 baseline and Whisper small.en. It does not train or guarantee speaker accuracy.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError, distributions

wanted = {'numpy':'2.2.6', 'pyannote.audio':'4.0.7', 'faster-whisper':'1.2.1'}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None

if all(installed(k) == v for k,v in wanted.items()):
    print('Libraries ready. Continue below.')
else:
    pins = dict(wanted)
    for name in ('torch','torchaudio','torchvision'):
        if installed(name): pins[name] = installed(name)
    for dist in distributions():
        name = dist.metadata.get('Name', '').lower().replace('_','-')
        if name.startswith('opentelemetry-'): pins[name] = dist.version
    constraints = Path('/kaggle/working/voxmeet-demo-constraints.txt')
    constraints.write_text('\n'.join(f'{k}=={v}' for k,v in pins.items()))
    env = os.environ.copy()
    env.pop('PIP_CONSTRAINT', None)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
                    *[f'{k}=={v}' for k,v in wanted.items()], 'jedi==0.19.2'],
                   env=env, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'check'], env=env, check=False)
    print('Setup complete. Run → Restart & clear cell outputs, then run the next cell.')


In [ ]:
import json, threading, time, traceback, textwrap
import requests
from kaggle_secrets import UserSecretsClient

WORKER_URL = WORKER_URL_PLACEHOLDER
WORKER_KEY = WORKER_KEY_PLACEHOLDER
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
HEADERS = {'Authorization': 'Bearer ' + WORKER_KEY}
BASE = WORKER_URL.split('/worker/')[0]
stop = threading.Event()

def heartbeat():
    while not stop.is_set():
        try:
            r = requests.post(WORKER_URL+'/heartbeat', headers=HEADERS, timeout=15)
            if r.status_code in (401, 403, 410):
                stop.set()
        except requests.RequestException:
            pass
        stop.wait(20)

def process(raw_zip):
    import gc, hashlib, io, json, time, zipfile
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from huggingface_hub import snapshot_download
    from pyannote.audio import Pipeline
    from pyannote.audio.pipelines.utils.hook import ProgressHook

    assert torch.cuda.is_available(), 'Enable T4 GPU first.'
    with zipfile.ZipFile(io.BytesIO(raw_zip)) as z:
        assert set(z.namelist()) == {'audio.wav', 'job.json'}, 'Not a VoxMeet audio package.'
        assert z.getinfo('audio.wav').file_size <= 120 * 1024 * 1024
        assert z.getinfo('job.json').file_size < 10000
        job = json.loads(z.read('job.json'))
        raw_audio = z.read('audio.wav')
    assert job['schema'] == 'voxmeet-gpu-v1'
    assert hashlib.sha256(raw_audio).hexdigest() == job['audio_sha256']
    samples, sr = sf.read(io.BytesIO(raw_audio), dtype='float32', always_2d=True)
    assert sr == 16000 and samples.shape[1] == 1 and np.isfinite(samples).all()
    samples = samples[:, 0].copy()
    assert 0 < len(samples) / sr <= 1801
    revision = '3533c8cf8e369892e6b79ff1bf80f7b0286a54ee'
    assert job['model_revision'] == revision
    print(f'Audio ready: {len(samples)/sr:.2f}s. Recognizing English...', flush=True)
    started = time.perf_counter()
    # Run Whisper in an isolated process with CUDA 12 runtime libraries.
    # PyTorch/pyannote keep their existing CUDA runtime in the notebook process.
    import os, subprocess, sys
    cuda_root = Path('/kaggle/working/voxmeet_cuda12')
    required_libs = ('libcublas.so.12', 'libcublasLt.so.12', 'libcudnn.so.9', 'libcudart.so.12', 'libnvrtc.so.12')
    if not all(list(cuda_root.rglob(name)) for name in required_libs):
        print('Installing isolated CUDA 12 libraries for Whisper...', flush=True)
        install_env = os.environ.copy()
        install_env.pop('PIP_CONSTRAINT', None)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-deps',
                        '--target', str(cuda_root), 'nvidia-cublas-cu12',
                        'nvidia-cudnn-cu12>=9,<10', 'nvidia-cuda-runtime-cu12',
                        'nvidia-cuda-nvrtc-cu12'], env=install_env, check=True)
    lib_dirs = sorted({str(p.parent) for p in cuda_root.rglob('*.so*')})
    assert all(list(cuda_root.rglob(name)) for name in required_libs), 'CUDA 12 library installation is incomplete.'
    asr_env = os.environ.copy()
    asr_env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [asr_env.get('LD_LIBRARY_PATH', '')])
    work = Path('/kaggle/working/voxmeet_asr_job')
    work.mkdir(exist_ok=True)
    np.save(work / 'audio.npy', samples)
    worker = work / 'recognize.py'
    worker.write_text(textwrap.dedent("""
    import ctypes, json, sys
    from pathlib import Path
    import numpy as np
    for name in ('libcublas.so.12', 'libcudnn.so.9'):
        ctypes.CDLL(name)
    from faster_whisper import WhisperModel
    work = Path(sys.argv[1])
    samples = np.load(work / 'audio.npy')
    model = WhisperModel('small.en', device='cuda', compute_type='float16')
    parts, _ = model.transcribe(samples, language='en', beam_size=5,
                                word_timestamps=True, vad_filter=True,
                                condition_on_previous_text=False)
    words = []
    for index, part in enumerate(parts):
        words.extend(dict(start=float(w.start), end=float(w.end), word=w.word,
                          confidence=float(w.probability)) for w in (part.words or []))
        if index % 10 == 0: print(f'ASR: {part.end:.1f}/{len(samples)/16000:.1f}s', flush=True)
    (work / 'words.json').write_text(json.dumps(words), encoding='utf-8')
    """), encoding='utf-8')
    subprocess.run([sys.executable, str(worker), str(work)], env=asr_env, check=True)
    words = json.loads((work / 'words.json').read_text(encoding='utf-8'))

    gc.collect(); torch.cuda.empty_cache()
    assert words, 'No speech detected. Try clearer audio; no transcript will be replaced.'
    print('Loading Community-1 baseline...', flush=True)
    model_dir = snapshot_download('pyannote/speaker-diarization-community-1',
                                  revision=revision, token=HF_TOKEN)
    pipeline = Pipeline.from_pretrained(model_dir).to(torch.device('cuda'))
    with ProgressHook() as hook:
        output = pipeline({'waveform':torch.from_numpy(samples[None,:]), 'sample_rate':sr}, hook=hook, **({'num_speakers': job['num_speakers']} if job.get('num_speakers') else {}))
    turns = [dict(start=float(t.start), end=float(t.end), speaker=label)
             for t, _, label in output.speaker_diarization.itertracks(yield_label=True)]
    from collections import defaultdict
    def overlap(a, b):
        return max(0, min(a['end'], b['end']) - max(a['start'], b['start']))

    def group_words(words):
        segments = []
        for word in words:
            text = word.get('punctuated_word', word.get('word', '')).strip()
            if not text:
                continue
            speaker = word.get('speaker')
            if speaker is not None:
                speaker = str(speaker)
            if (not segments or segments[-1]['speaker'] != speaker or
                    word['start'] - segments[-1]['end'] > 2.5 or
                    word['end'] - segments[-1]['start'] > 30):
                segments.append(dict(id=f's{len(segments)+1}', speaker=speaker,
                                     start=float(word['start']), end=float(word['end']), text=text, words=[]))
            else:
                segments[-1]['text'] += ' ' + text
                segments[-1]['end'] = max(segments[-1]['end'], float(word['end']))
            segments[-1]['words'].append(dict(text=text, start=float(word['start']), end=float(word['end']),
                                             confidence=word.get('confidence')))
        return segments

    def assign_speakers(words, turns):
        for word in words:
            # Whisper sometimes returns point timestamps for short words. Assign
            # only when exactly one speaker is active at that point.
            if word['start'] == word['end']:
                speakers = {turn['speaker'] for turn in turns if turn['start'] <= word['start'] < turn['end']}
                word['speaker'] = next(iter(speakers)) if len(speakers) == 1 else None
                continue
            # A voice can have several adjacent intervals within one word. Compare
            # total covered time per voice, merging intervals to avoid double count.
            intervals = defaultdict(list)
            for turn in turns:
                if overlap(word, turn) > 0:
                    intervals[turn['speaker']].append((max(word['start'], turn['start']),
                                                       min(word['end'], turn['end'])))
            scores = {}
            for speaker, spans in intervals.items():
                total, right = 0., float('-inf')
                for start, end in sorted(spans):
                    total += max(0., end - max(start, right))
                    right = max(right, end)
                scores[speaker] = total
            ranked = sorted(scores, key=scores.get, reverse=True)
            word['speaker'] = ranked[0] if ranked and (len(ranked) == 1 or
                scores[ranked[0]] - scores[ranked[1]] > 1e-6) else None
            word['ambiguous_speaker'] = len(ranked) > 1
        return words

    utterances = group_words(assign_speakers(words, turns))
    for index, utterance in enumerate(utterances, 1):
        utterance['id'] = f'U{index:04d}'
    result = dict(job, utterances=utterances, words=words, diarization_turns=turns, provenance=dict(
        asr_model='Whisper small.en',diarization_model='Community-1 baseline',
        diarization_revision=revision), processing_seconds=round(time.perf_counter()-started,1))
    result_path = Path('/kaggle/working') / f"voxmeet-result-{job['recording_id']}.json"
    result_path.write_text(json.dumps(result,ensure_ascii=False), encoding='utf-8')
    print('Done:',len(utterances),'turns |',len({t['speaker'] for t in turns}),'speaker groups')
    print('Unassigned words:',sum(w['speaker'] is None for w in words))
    print('Review speaker labels in the local demo before summarizing.')
    return result_path.read_bytes()

thread = threading.Thread(target=heartbeat, daemon=True)
thread.start()
print('Worker running. Choose Community-1 in your local demo. Stop this cell to disconnect.')
try:
    while not stop.is_set():
        try:
            response = requests.get(WORKER_URL+'/next', headers=HEADERS, timeout=30)
            if response.status_code in (401, 403, 410):
                print('Connection expired. Reconnect from the demo.'); break
            response.raise_for_status()
            job_id = response.json()['job']
        except requests.RequestException:
            print('Connection interrupted; retrying in 10 seconds.')
            stop.wait(10); continue
        if not job_id:
            stop.wait(3); continue
        endpoint = BASE + '/exchange/' + job_id
        try:
            response = requests.get(endpoint+'/audio', headers=HEADERS, timeout=180)
            response.raise_for_status()
            result = process(response.content)
        except Exception as exc:
            # Do not print request URLs containing connection identifiers.
            print('Processing failed:', type(exc).__name__)
            if not isinstance(exc, requests.RequestException): traceback.print_exc()
            try: requests.post(endpoint+'/failed', headers=HEADERS, timeout=20)
            except requests.RequestException: pass
            stop.wait(5); continue
        # Retry result transfer without rerunning inference. Server writes idempotently.
        for attempt in range(12):
            try:
                response = requests.post(endpoint+'/result', headers=HEADERS, data=result, timeout=120)
                if response.status_code in (401, 403, 409, 410):
                    print('Job closed; result remains in /kaggle/working.'); break
                response.raise_for_status()
                print('Transcript returned automatically. Waiting for the next recording.'); break
            except requests.RequestException:
                stop.wait(5)
        else:
            print('Could not return result. Stopping to avoid reprocessing this audio.'); break
finally:
    stop.set()
    thread.join(timeout=20)
    print('Worker stopped. Audio and results in /kaggle/working are temporary until saved.')
